# Prepare data from 

1. Get data
* acces data from samba server by first mounting it to the sciencecloud VM (befoe:`sudo apt install cifs-utils`): 
`sudo mount -t cifs //idnas37.d.uzh.ch/G_ADABD_Largefiles$ /mnt_AdaBD_largefiles -o username=mrenke,password=St2689oker_,uid=$(id -u),gid=$(id -g)`
* copy to  bids_folder
    * rsync -ah --progress /mnt_AdaBD_largefiles/Data/SMILE_Data/measurements/mri_files/*.zip /mnt_03/ds-smile1/sourcedata/
    * unzip into own folder
        `for zipfile in *.zip; do 
            foldername="${zipfile%.zip}"
            mkdir -p "$foldername"
            unzip "$zipfile" -d "$foldername"
        done`
* unzip directlly into new bids_folder/sourcedata:
    * go into `/mnt_AdaBD_largefiles/Data/SMILE_Data/measurements/mri_files`
    `target_folder="/mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-smile/sourcedata"`
    * `for zipfile in *.zip; do 
            foldername="${target_folder}/${zipfile%.zip}"
            mkdir -p "$foldername"
            unzip "$zipfile" -d "$foldername"
        done`        
2. Copy and Rename files --> just keep 3 digit sub-ID (first digiyt implies treatment arm/group)

* combine for some instances 3D func into one 4D file

3. Check BIDS format via command line:
`deno run -ERN jsr:@bids/validator /mnt_03/ds-smile1` (via deno, installed: `curl -fsSL https://deno.land/install.sh | sh`)
warning:  NIFTI_PIXDIM NIfTI file's header field for pixel dimension information is empty or too short.
error:
* fix pixdimm (TR) in file header, careful, TR(rest) != TR(magjudge)
* exception sub with double 4D scan


- Caroline:"Mir ist gester aufgefallen, dass das mit der TR  vom resting state nicht stimmt. Beim scannen habe ich auf den wert geschaut und gemerkt, dass er höher ist, weil es sonst eine fehlermeldung gibt. Ich kann dir gerne heute mal den korrekten wert weiterleiten." --> Es wird immer auf 2035.0 angepasst


In [9]:
bids_folder = '/mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-smile' 
import os
subList = [int(sub[4:]) for sub in os.listdir(bids_folder) if sub.startswith('sub-')]
print(*subList, sep=',')

101,102,103,104,105,106,107,108,109,110,111,114,116,117,118,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,302,303,305,306,307,308,310,311,312,314,315,316


# Problems detected after fmriprep

for sub-102 & 201
`ValueError: Whoops, not enough data in file`



In [1]:
import os
import shutil
import re
import nibabel as nib

In [2]:
# Define source and destination paths

source_path = '/mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-smile/sourcedata' #"/mnt_03/ds-smile1/sourcedata"  # Adjust this to your source directory
dest_path = '/mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-smile'    # Adjust this to your target BIDS directory


# Function to create BIDS-compliant folder structure
def create_bids_structure(sub_id, ses_id, modality):
    bids_sub_path = os.path.join(dest_path, f"sub-{sub_id}")
    bids_ses_path = os.path.join(bids_sub_path, f"ses-{ses_id}")
    bids_modality_path = os.path.join(bids_ses_path, modality)
    os.makedirs(bids_modality_path, exist_ok=True)
    return bids_modality_path

# Map source folder names to BIDS target task names
task_mapping = {
    "NumRisk_R1": "task-magjudge_run-1",
    "NumRisk_R2": "task-magjudge_run-2",
    "NumRisk_R3": "task-magjudge_run-3",
    "PlaceValue": "task-placevalue_run-1",
    "rsfMRI": "task-rest_run-1",
    "T1": "T1w"
}

def zero_pad(id_str, length=2):
    return id_str.zfill(length)

In [5]:
# Iterate through subjects and sessions in the source directory
for entry in os.scandir(source_path):
    if entry.is_dir() and entry.name.startswith("SMILE"):
        # Extract subject ID and session ID
        match = re.match(r"SMILE(\d{3})_d(\d+)", entry.name)
        if match:
            sub_id = match.group(1)
            ses_id = match.group(2)

            # scip if already exists:
            modality_type = "func" # example to check (anat only for ses 1 !)
            bids_path = create_bids_structure(sub_id, ses_id, modality_type)
            if os.listdir(bids_path): # somehow seems to work and returns True/False
                print(f"Data for sub-{sub_id} ses-{ses_id} already exists in BIDS format. Skipping...")
            else:
                # Iterate through modality folders
                for modality_folder in os.scandir(entry.path):
                    if modality_folder.is_dir():
                        source_modality_name = modality_folder.name
                        target_modality_name = task_mapping.get(source_modality_name, None)

                        if target_modality_name:
                            # Create BIDS structure
                            modality_type = "func" if "task" in target_modality_name else "anat"
                            bids_path = create_bids_structure(sub_id, ses_id, modality_type)

                            # Process files in the modality folder
                            for file in os.scandir(modality_folder.path):
                                if file.is_file():
                                    # Extract scan order from the filename
                                    scan_order_match = re.match(r"(\d{3})_.*\.(nii|json)", file.name) # names are like 001_*.nii
                                    if scan_order_match:
                                        scan_order = zero_pad(scan_order_match.group(1))

                                        # Build BIDS filename
                                        if target_modality_name == "T1w":
                                            bids_filename = f"sub-{sub_id}_ses-{ses_id}_T1w.{scan_order_match.group(2)}"
                                        else:
                                            bids_filename = f"sub-{sub_id}_ses-{ses_id}_{target_modality_name}_bold.{scan_order_match.group(2)}"

                                        # Copy file to BIDS structure
                                        shutil.copy(file.path, os.path.join(bids_path, bids_filename))

            print(f"Data of sub-{sub_id} successfully organized into BIDS format.")


Data for sub-101 ses-1 already exists in BIDS format. Skipping...
Data of sub-101 successfully organized into BIDS format.
Data for sub-101 ses-2 already exists in BIDS format. Skipping...
Data of sub-101 successfully organized into BIDS format.
Data for sub-102 ses-1 already exists in BIDS format. Skipping...
Data of sub-102 successfully organized into BIDS format.
Data for sub-102 ses-2 already exists in BIDS format. Skipping...
Data of sub-102 successfully organized into BIDS format.
Data for sub-103 ses-1 already exists in BIDS format. Skipping...
Data of sub-103 successfully organized into BIDS format.
Data for sub-103 ses-2 already exists in BIDS format. Skipping...
Data of sub-103 successfully organized into BIDS format.
Data for sub-104 ses-1 already exists in BIDS format. Skipping...
Data of sub-104 successfully organized into BIDS format.
Data for sub-104 ses-2 already exists in BIDS format. Skipping...
Data of sub-104 successfully organized into BIDS format.
Data for sub-105

In [ ]:
# for the func runs where each volume is a single 3D nifti file, we need to combine them into a single 4D nifti file

import nibabel as nib
import numpy as np

for entry in os.scandir(source_path):
    if entry.is_dir() and entry.name.startswith("SMILE"):
        # Extract subject ID and session ID
        match = re.match(r"SMILE(\d{3})_d(\d+)", entry.name)
        if match:
            sub_id = match.group(1)
            ses_id = match.group(2)

            # Iterate through modality folders
            for modality_folder in os.scandir(entry.path):
                if modality_folder.is_dir():
                    source_modality_name = modality_folder.name
                    target_modality_name = task_mapping.get(source_modality_name, None)
                    
                    if target_modality_name is not None and "task" in target_modality_name:
                        modality_type = "func" 
                        bids_path = create_bids_structure(sub_id, ses_id, modality_type)
                        bids_filename = f"sub-{sub_id}_ses-{ses_id}_{target_modality_name}_bold.nii"

                        source_folder = modality_folder.path
                        nifti_files = sorted([os.path.join(source_folder, f) for f in os.listdir(source_folder) if f.endswith(".nii")])
                        if len(nifti_files) > 10: # some weird sub has twice the same func run in 4D
                            #print(f'sub-{sub_id} ses-{ses_id} {source_modality_name} is saved as mutiple 3D files')
                            volumes = []
                            for file in nifti_files:
                                img = nib.load(file)
                                volumes.append(img.get_fdata())
                            affine = nib.load(nifti_files[0]).affine
                            header = nib.load(nifti_files[0]).header
                            combined_img = nib.Nifti1Image(np.stack(volumes, axis=-1), affine, header)
                            
                            output_file = os.path.join(bids_path, bids_filename)
                            nib.save(combined_img, output_file)
                            print(f'sub-{sub_id} ses-{ses_id} {source_modality_name} now saved as 4D nifit file!')

                        



In [36]:
sub= '201'
pati = f'/mnt_03/ds-smile1/sub-{sub}/ses-1/func/'

nifti_file = os.path.join(pati, f'sub-{sub}_ses-1_task-magjudge_run-3.nii')
img = nib.load(nifti_file)
img.header['pixdim']

array([-1.       ,  1.8750001,  1.875    ,  2.5000002,  0.       ,
        1.       ,  1.       ,  1.       ], dtype=float32)

In [34]:
sub= '102'
pati = f'/mnt_03/ds-smile1/sub-{sub}/ses-1/func/'

nifti_file = os.path.join(pati, f'sub-{sub}_ses-1_task-magjudge_run-1.nii')
img = nib.load(nifti_file)
img.header['pixdim']

array([-1.   ,  1.875,  1.875,  2.5  ,  2.6  ,  0.   ,  0.   ,  0.   ],
      dtype=float32)

`[ERROR] NOT_INCLUDED Files with such naming scheme are not part of BIDS specification. This error is most commonly caused by typos in file names that make them not BIDS compatible. Please consult the specification and make sure your files are named correctly. If this is not a file naming issue (for example when including files not yet covered by the BIDS specification) you should include a ".bidsignore" file in your dataset (see https://github.com/bids-standard/bids-validator#bidsignore for details). Please note that derived (processed) data should be placed in /derivatives folder and source data (such as DICOMS or behavioural logs in proprietary formats) should be placed in the /sourcedata folder.
		/sub-201/ses-1/func/sub-201_ses-1_task-magjudge_run-1.json
		/sub-201/ses-1/func/sub-201_ses-1_task-magjudge_run-3.nii`

In [ ]:
import os
import re

# Set the root BIDS folder (replace with your actual BIDS folder path)
bids_root = "/mnt_03/ds-smile1"

# Function to rename files in the func folder
def rename_func_files(bids_root):
    # Walk through all subfolders
    for root, dirs, files in os.walk(bids_root):
        # Only focus on 'func' subfolders
        if 'func' in root:
            for file in files:
                # Check if the file is a .nii or .json file
                if file.endswith('.nii') or file.endswith('.json'):
                    # Check if the file is a functional run (looking for task names in the filename)
                    if re.search(r'_task-[a-zA-Z0-9]+_', file):
                        # Construct the new name by adding '_bold' before the extension
                        new_name = re.sub(r'(.*_task-[a-zA-Z0-9]+)_run-(\d+)(\.[a-zA-Z]+)', r'\1_run-\2_bold\3', file)
                        
                        # Get the full paths for old and new names
                        old_path = os.path.join(root, file)
                        new_path = os.path.join(root, new_name)
                        
                        # Rename the file if the names are different
                        if old_path != new_path:
                            print(f"Renaming: {old_path} --> {new_path}")
                            os.rename(old_path, new_path)

# Run the renaming function
rename_func_files(bids_root)


# TR time error
	[ERROR] REPETITION_TIME_MISMATCH Repetition time did not match between the scan's header and the associated JSON metadata file.

		/sub-201/ses-1/func/sub-201_ses-1_task-magjudge_run-3_bold.nii
		/sub-201/ses-1/func/sub-201_ses-1_task-rest_run-1_bold.nii

In [3]:
# NIfTI file that seems file
sub= '101' # wrong : '201'
pati = f'/mnt_03/ds-smile1/sub-{sub}/ses-1/func/'
nifti_file = os.path.join(pati, f'sub-{sub}_ses-1_task-magjudge_run-1_bold.nii')# Load the NIfTI image
img = nib.load(nifti_file)
# Extract the TR from the header
tr = img.header.get_zooms()[3]  # TR is the 4th element in the zooms tuple

print(f"TR in NIfTI header: {tr}")
print(img.affine)
print(img.header)

TR in NIfTI header: 2.5999999046325684
[[-1.87133038e+00 -4.17411327e-03  1.56234786e-01  1.17629280e+02]
 [ 0.00000000e+00  1.87381160e+00  8.89981613e-02 -1.20023994e+02]
 [ 1.17250375e-01 -6.66179657e-02  2.49352646e+00 -4.91707153e+01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
<class 'nibabel.nifti1.Nifti1Header'> object, endian='<'
sizeof_hdr      : 348
data_type       : b''
db_name         : b''
extents         : 0
session_error   : 0
regular         : b'r'
dim_info        : 57
dim             : [  4 128 128  42 166   1   1   1]
intent_p1       : 0.0
intent_p2       : 0.0
intent_p3       : 0.0
intent_code     : none
datatype        : int16
bitpix          : 16
slice_start     : 0
pixdim          : [-1.     1.875  1.875  2.5    2.6    0.     0.     0.   ]
vox_offset      : 0.0
scl_slope       : nan
scl_inter       : nan
slice_end       : 0
slice_code      : unknown
xyzt_units      : 10
cal_max         : 0.0
cal_min         : 0.0
slice_duration  : 0.0
toff

In [4]:
nifti_file = os.path.join(pati, f'sub-{sub}_ses-1_task-magjudge_run-1_bold.nii')
img = nib.load(nifti_file)
pixdim_target_array_magjudge = img.header['pixdim']
pixdim_target_array_magjudge

array([-1.   ,  1.875,  1.875,  2.5  ,  2.6  ,  0.   ,  0.   ,  0.   ],
      dtype=float32)

In [5]:
nifti_file = os.path.join(pati, f'sub-{sub}_ses-1_task-rest_run-1_bold.nii')
img = nib.load(nifti_file)
print(img.header.get_zooms()[3])
pixdim_target_array_rest = img.header['pixdim']
pixdim_target_array_rest

2.035


array([-1.   ,  1.875,  1.875,  2.5  ,  2.035,  0.   ,  0.   ,  0.   ],
      dtype=float32)

In [6]:
bids_folder = '/mnt_03/ds-smile1/'
subList = [f[4:7] for f in os.listdir(os.path.join(bids_folder)) if f[0:4] == 'sub-' and len(f)==7]
sesList = [1,2]
tasks = {
    'rest': range(1, 2),         # Only run 1 for 'rest'
    #'magjudge': range(1, 4)    
}


In [17]:
np.round(tr,3) != np.round(pixdim_target_array[4],3)

False

In [14]:
np.round(pixdim_target_array[4],3)

2.035

In [18]:
import numpy as np

for sub in subList:
    for ses in sesList:
        pati = f'/mnt_03/ds-smile1/sub-{sub}/ses-{ses}/func/'

        for task, runs in tasks.items():
            pixdim_target_array = pixdim_target_array_rest if task == 'rest' else pixdim_target_array_magjudge
            for run in runs:
                print(f"checking sub-{sub}, ses-{ses}, task-{task}, run-{run}")
                nifti_file = os.path.join(pati, f'sub-{sub}_ses-{ses}_task-{task}_run-{run}_bold.nii')
                if os.path.exists(nifti_file):
                    img = nib.load(nifti_file)
                    tr = img.header.get_zooms()[3]
                    if np.round(tr,3) != np.round(pixdim_target_array[4],3):
                        print(f"TR is {tr} for sub-{sub}, run-{run}. Fixing pixdim...")
                                    
                        # Update the pixdim field in the header
                        header = img.header
                        header['pixdim'] = pixdim_target_array
                        
                        # Save the updated NIfTI image
                        nib.save(img, nifti_file)
                        print(f"Fixed pixdim for sub-{sub}, task-{task}, run-{run}.")

checking sub-201, ses-1, task-rest, run-1
checking sub-201, ses-2, task-rest, run-1
checking sub-311, ses-1, task-rest, run-1
checking sub-311, ses-2, task-rest, run-1
checking sub-303, ses-1, task-rest, run-1
checking sub-303, ses-2, task-rest, run-1
checking sub-102, ses-1, task-rest, run-1
checking sub-102, ses-2, task-rest, run-1
TR is 2.5999999046325684 for sub-102, run-1. Fixing pixdim...
Fixed pixdim for sub-102, task-rest, run-1.
checking sub-103, ses-1, task-rest, run-1
TR is 2.5999999046325684 for sub-103, run-1. Fixing pixdim...
Fixed pixdim for sub-103, task-rest, run-1.
checking sub-103, ses-2, task-rest, run-1
TR is 2.5999999046325684 for sub-103, run-1. Fixing pixdim...
Fixed pixdim for sub-103, task-rest, run-1.
checking sub-203, ses-1, task-rest, run-1
TR is 2.5999999046325684 for sub-203, run-1. Fixing pixdim...
Fixed pixdim for sub-203, task-rest, run-1.
checking sub-203, ses-2, task-rest, run-1
checking sub-105, ses-1, task-rest, run-1
TR is 2.5999999046325684 for s

In [96]:
# Load NIfTI file
sub= '311'
run = 1
task ='rest'
pati = f'/mnt_03/ds-smile1/sub-{sub}/ses-1/func/'
nifti_file = os.path.join(pati, f'sub-{sub}_ses-1_task-{task}_run-{run}_bold.nii')# Load the NIfTI image
img = nib.load(nifti_file)
# Extract the TR from the header
tr = img.header.get_zooms()[3]  # TR is the 4th element in the zooms tuple

print(f"TR in NIfTI header: {tr}")
print(img.affine)
print(img.header)

TR in NIfTI header: 2.5999999046325684
[[-1.87181020e+00  1.34539604e-03  1.45754158e-01  1.22654617e+02]
 [ 1.72287226e-03  1.87498820e+00  8.56500864e-03 -1.14338615e+02]
 [ 1.09310627e-01 -6.51335716e-03  2.49573302e+00 -2.29963875e+01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
<class 'nibabel.nifti1.Nifti1Header'> object, endian='<'
sizeof_hdr      : 348
data_type       : b''
db_name         : b''
extents         : 0
session_error   : 0
regular         : b'r'
dim_info        : 0
dim             : [  4 128 128  42 173   1   1   1]
intent_p1       : 0.0
intent_p2       : 0.0
intent_p3       : 0.0
intent_code     : none
datatype        : int16
bitpix          : 16
slice_start     : 0
pixdim          : [-1.     1.875  1.875  2.5    2.6    0.     0.     0.   ]
vox_offset      : 0.0
scl_slope       : nan
scl_inter       : nan
slice_end       : 0
slice_code      : unknown
xyzt_units      : 10
cal_max         : 0.0
cal_min         : 0.0
slice_duration  : 0.0
toffs

In [93]:
img.header.get_zooms()
img.header['pixdim']

array([-1.   ,  1.875,  1.875,  2.5  ,  2.6  ,  0.   ,  0.   ,  0.   ],
      dtype=float32)

	[ERROR] BOLD_NOT_4D BOLD scans must be 4 dimensional.

		/sub-202/ses-1/func/sub-202_ses-1_task-magjudge_run-1_bold.nii


--> twice the same file! 
`cmp --silent 004_MR_Ax_fMRI_NumRisk_R1_SMILE_Ax_fMRI_NumRisk_R1_EP_GR.nii 004_MR_Ax_fMRI_NumRisk_R1_SMILE_Ax_fMRI_NumRisk_R1_EP_GRa.nii || echo "files are different"` retruns nothing!!

In [61]:
# twice the same file!


sub_id = '202'
ses_id = 1
run = 1

source_modality_name = f'NumRisk_R{run}'
target_modality_name = task_mapping.get(source_modality_name, None)

modality_folder_path = f'/mnt_03/ds-smile1/sourcedata/SMILE{sub_id}_d{ses_id}/{source_modality_name}'

# Create BIDS structure
modality_type = "func" if "task" in target_modality_name else "anat"
bids_path = create_bids_structure(sub_id, ses_id, modality_type)

# Process files in the modality folder
for file in os.scandir(modality_folder_path):
    print(file.name)
    scan_order_match = re.match(r"(\d{3})_.*_GR\.(nii|json)$", file.name) # take only file ending with GR and not also GRa
    if scan_order_match:
        print(scan_order_match)
        # Build BIDS filename
        bids_filename = f"sub-{sub_id}_ses-{ses_id}_{target_modality_name}_bold.{scan_order_match.group(2)}"
        # Copy file to BIDS structure
        shutil.copy(file.path, os.path.join(bids_path, bids_filename))

004_MR_Ax_fMRI_NumRisk_R1_SMILE_Ax_fMRI_NumRisk_R1_EP_GR.json
<re.Match object; span=(0, 61), match='004_MR_Ax_fMRI_NumRisk_R1_SMILE_Ax_fMRI_NumRisk_R>
004_MR_Ax_fMRI_NumRisk_R1_SMILE_Ax_fMRI_NumRisk_R1_EP_GRa.json
ZfMRF_DicomTK.log
004_MR_Ax_fMRI_NumRisk_R1_SMILE_Ax_fMRI_NumRisk_R1_EP_GRa.nii
004_MR_Ax_fMRI_NumRisk_R1_SMILE_Ax_fMRI_NumRisk_R1_EP_GR.nii
<re.Match object; span=(0, 60), match='004_MR_Ax_fMRI_NumRisk_R1_SMILE_Ax_fMRI_NumRisk_R>


# Problems detected after fmriprep

for sub-102 & 201
`ValueError: Whoops, not enough data in file`

* 102 - rest
* 201 - magjudge


In [39]:
sub = 201
ses = 1
task = 'magjudge'
run = 3
pati = f'/mnt_03/ds-smile1/sub-{sub}/ses-{ses}/func/'
nifti_file = os.path.join(pati, f'sub-{sub}_ses-{ses}_task-{task}_run-{run}_bold.nii')# Load the NIfTI image
img = nib.load(nifti_file)
img.get_data()

/tmp/ipykernel_2921050/635943318.py:8: DeprecationWarning: get_data() is deprecated in favor of get_fdata(), which has a more predictable return type. To obtain get_data() behavior going forward, use numpy.asanyarray(img.dataobj).

* deprecated from version: 3.0
* Will raise <class 'nibabel.deprecator.ExpiredDeprecationError'> as of version: 5.0
  img.get_data()


array([[[[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],

        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],

        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],

        ...,

        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
    

In [ ]:
sub = 101
task_folder = ['rsfMRI', 'NumRisk_R2', 'T1', 'NumRisk_R3', 'NumRisk_R1']
pati = f'{source_path}/SMILE{sub}_d{ses}/{task_folder[0]}'
file = os.listdir(pati)[0]
img = nib.load(f'{pati}/{file}') # nifti is first

img.get_fdata()

memmap([[[[0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          ...,
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.]],

         [[0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          ...,
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.]],

         [[0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          ...,
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.]],

         ...,

         [[0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          ...,
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0.

In [31]:
sub = 201
ses = 1
run = 1
task = 'magjudge'

task_folder = 'rsfMRI' if task == 'rest' else f'NumRisk_R{run}'
pati = f'{source_path}/SMILE{sub}_d{ses}/{task_folder}'
file = [file for file in os.listdir(pati) if file.endswith('.nii')][0]

source_file_path = f'{pati}/{file}'

pati = f'/mnt_03/ds-smile1/sub-{sub}/ses-{ses}/func/'
target_file_path_renamed = os.path.join(pati, f'sub-{sub}_ses-{ses}_task-{task}_run-{run}_bold.nii')
shutil.copy(source_file_path, target_file_path_renamed)

'/mnt_03/ds-smile1/sub-201/ses-1/func/sub-201_ses-1_task-magjudge_run-1_bold.nii'

In [30]:
task_folder = 'rsfMRI' if task == 'rest' else f'NumRisk_R{run}'
pati = f'{source_path}/SMILE{sub}_d{ses}/{task_folder}'
os.listdir(pati)


['004_MR_Ax_fMRI_NumRisk_R1_SMILE_Ax_fMRI_NumRisk_R1_EP_RM.json',
 'ZfMRF_DicomTK.log',
 '004_MR_Ax_fMRI_NumRisk_R1_SMILE_Ax_fMRI_NumRisk_R1_EP_RM.nii']

In [32]:
nib.load(target_file_path_renamed).get_fdata()

memmap([[[[0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          ...,
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.]],

         [[0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          ...,
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.]],

         [[0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          ...,
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.]],

         ...,

         [[0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          ...,
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0.